*   Weight decay is a regularization method that discourages large weights.
*   It helps control overfitting by adding a size penalty to the training objective.


In [3]:
import math
import random
import numpy as np
import torch

# 3.7.1 Norms and Weight Decay

## 1. Intuition

*   A norm measures the size of a vector.
> *   The L2 norm squares each value, sums the squares, and takes a square root.
*   Weight decay penalizes large weights by adding a term based on weight size to the loss.

## 2. Why this exists

* Large weights can make a model overly sensitive to small input changes.
> * If those large weights arise from fitting noise in the training data, the model may overfit and perform poorly on unseen data.
> * Regularization penalizes large weights, encouraging simpler functions that are typically more robust and generalize better.
* Conversly, penalizing weight size encourages simpler functions.

## 3. Examples

*   Manual L2 penalty for a small weight vector.

In [4]:
w = [3.0, 4.0]
l2_squared = sum(value * value for value in w) # 3*3 + 4*4 = 9 + 16 = 25
penalty = 0.5 * l2_squared # 25 / 2 = 12.5
penalty

12.5

*   PyTorch L2 penalty.

In [5]:
w = torch.tensor([3.0, 4.0])
penalty = 0.5 * torch.sum(w ** 2) # Equivalent of w ^ 2
penalty

tensor(12.5000)

## 4. Step-by-step breakdown

*   The manual code squares each weight and sums the results.

*   The factor `0.5` is a convenience because it simplifies derivatives.

*   In PyTorch, `w ** 2` squares each weight elementwise.

*   `torch.sum(...)` reduces all squared values into one scalar penalty.

## 5. Connection to ML systems

*   Weight decay modifies the training objective from `data loss` to `data loss + weight-size penalty`, effectively forcing the weight to be as small as possible.
*   The model therefore balances fitting the training data with keeping the weights small, often leading to simpler models that generalize better.

## 6. Common confusion points

- Weight decay usually penalizes weights, not the bias.
- The penalty is added during training, not after training.
> * The penalty term is included in the loss function at every training step.
> * Since the optimizer minimizes the combined loss, not just the data loss.
> * The penalty strength (λ) is a hyperparameter that determines how strongly large weights are penalized.
- A larger penalty strength forces smaller weights more strongly.
- Smaller weights do not automatically mean better predictions
> * They help reduce the risk of overfitting to the training data, often improving generalization.

# 3.7.2 High-Dimensional Linear Regression

## 1. Intuition

*   High-dimensional data has many features.
*   If the number of features is large compared with the number of examples, overfitting becomes easier.
> *   A model with many features (and therefore often many parameters or weights) has more flexibility to fit not only the underlying patterns but also random noise in the training data.
> * This increased flexibility can increase the risk of overfitting.

## 2. Why this exists

*   Weight decay is especially useful when the model has enough flexibility to memorize the training data.

## 3. Examples

*   Create more features than examples.
* Because there are 20 features but only 5 training examples, even a simple linear model has enough flexibility to overfit by fitting both the underlying relationship and the random noise.
> * Put simply, our model with 20 input features is trying to learn 20 feature weights from only 5 training examples.
> * More degrees of freedom usually introduce a higher chance of overfitting, because the model can fit the training noise instead of learning the true pattern (in this case, the 2 features with nonzero true weights).

In [6]:
torch.manual_seed(0)
num_train = 5
num_features = 20

X = torch.randn(num_train, num_features) # Creates a shape of (5, 20)
true_w = torch.zeros(num_features) # Initialize 20 weights that are 0
true_w[:2] = torch.tensor([2.0, -3.4]) # Change the first 2 weights true_w[0, 1] to 2.0 and -3.4
y = X @ true_w + 0.01 * torch.randn(num_train) # Adds random Gaussian noise to y with torch.randn(num_train)

X.shape, y.shape # (5, 20) and (5, )

(torch.Size([5, 20]), torch.Size([5]))

## 4. Step-by-step breakdown

*   `num_features = 20` means each example has 20 input values.
*   `num_train = 5` means there are only 5 training examples.
*   Only the first 2 true weights are nonzero.
*   The model does not know this sparse structure unless we guide it with assumptions or regularization.

## 5. Connection to ML systems

*   Real high-dimensional problems include text features, genomics, recommendation systems, and wide tabular data.
*   Regularization helps reduce reliance on accidental patterns.

## 6. Common confusion points

- Many features increase the chance of fitting noise.
- More dimensions are not automatically bad, but they require more data (e.g. training examples) or regularization (e.g. weight decay).
- True useful features may be sparse.
- Synthetic high-dimensional data is useful for testing overfitting behavior.

# 3.7.3 Implementation from Scratch

## 1. Intuition

*   From scratch, weight decay is implemented by adding an L2 penalty to the loss before calling `backward()`.
*   The penalty strength is usually written as lambda (λ).
> * In code, use a name like `wd` because `lambda` is a Python keyword.

## 2. Why this exists

*   Adding the penalty manually shows exactly how regularization changes the objective and therefore the gradients.

## 3. Examples

*   Define the penalty and add it to a data loss.
*   Only weights are penalized, not the bias, since penalizing the bias can unnecessarily restrict the model's ability to shift predictions.
*   We use λ because different datasets/models can produce different loss and weight scales, so we need to tune how strongly we penalize large weights.

In [8]:
def l2_penalty(w):
  return 0.5 * torch.sum(w ** 2) # 1/2 is part of the penalty since the derivative of 1/2 would cancel out ^^2 (factor of 2); without the 1/2, the factor of 2 would simply be absorbed into λ

w = torch.randn(20, requires_grad=True) # Initialize weights at 20 random numbers (which fits X's shape of (5, 20))
b = torch.zeros(1, requires_grad=True) # Initialize bias with 1 zero
y_hat = X @ w + b

data_loss = ((y_hat - y) ** 2).mean() # Manual MSELoss
loss = data_loss + 0.1 * l2_penalty(w) # Apply our L2 regularization penalty via l2_penalty(w); 0.1 is the regularization strength (λ) to ratio the penalty itself
loss

tensor(19.5677, grad_fn=<AddBackward0>)

*   Compute gradients from the regularized objective.
*   Bias can be graded (have gradients computed) and updated too because it is a learnable parameter, just like weights.
> *   Even though it starts at a fixed value (e.g., 0), the optimizer can change it during training.
> *   B doesn't participate in l2 -> penalty -> regularization (all of which are for weights only), but bias still participates in backprop.
> *   We keep bias out of regularization because it's a controlling feature that shifts the prediction baseline, and penalizing it could unnecessarily restrict the model's ability to adjust its output offset.
> * If we also penalized the bias, we would make the model reluctant to change the bias value, which could hurt its ability to shift predictions to the correct baseline (like minimized to 0, which is usually not intended).

In [9]:
loss.backward() # Calculates gradients back to each params (the 20 weights and bias). To update them use step()
w.grad.shape, b.grad.shape

(torch.Size([20]), torch.Size([1]))

## 4. Step-by-step breakdown

*   `l2_penalty(w)` computes the size penalty for the weight vector.
*   `data_loss` measures prediction error.
*   `0.1 * l2_penalty(w)` scales the regularization strength.
*   The final `loss` combines prediction error and weight penalty.
*   Calling `backward()` computes gradients for the combined objective.



## 5. Connection to ML systems

*   This is the transparent version of weight decay.
*   Later optimizer options can apply the same idea more concisely.



## 6. Common confusion points

- Penalize `w`, not usually `b`.
- The regularization coefficient controls penalty strength.
- The penalty changes gradients, not just reported loss.
- Clear gradients before another backward pass in a real training loop.

# 3.7.4 Concise Implementation

## 1. Intuition

*   PyTorch optimizers can apply weight decay directly through an optimizer setting.
*   This keeps the loss expression focused on prediction error while the optimizer handles the weight penalty.

## 2. Why this exists

*   The concise option reduces boilerplate and matches common PyTorch practice.

## 3. Examples

*   Create an optimizer with weight decay.

In [10]:
net = torch.nn.Linear(20, 1) # Creates a layer of 20 input features and 1 output feature
optimizer = torch.optim.SGD(net.parameters(), lr=0.03, weight_decay=0.1) # Creates an SGD optimizer that updates all parameters in net using a learning rate of 0.03 and applies weight decay/regularization of 0.1
loss_fn = torch.nn.MSELoss() # Loss will be calculated via MSELoss()
y_col = y.reshape(-1, 1) # Reshape y to y_col so that it matches X (5, 20), it will be (5, 1)

optimizer.zero_grad() # Clear any-prexisting grads
loss = loss_fn(net(X), y_col) # Calculates loss using MSELoss() relatiev to y_col
loss.backward() # Calculates global gradients via loss backprop (chain-rule)
optimizer.step() # Update global parameters (all weights & bias) by using the gradients from loss backprop

loss

tensor(13.0660, grad_fn=<MseLossBackward0>)

## 4. Step-by-step breakdown

*   `torch.nn.Linear(20, 1)` creates a linear model for 20 features.

*   `weight_decay=0.1` tells the optimizer to include weight decay in parameter updates.

*   The explicit `loss` is only prediction loss here. It measures how well the current model predicts the targets.
> *   The penalty does not appear in the explicit loss because the optimizer handles weight decay during `optimizer.step()`.
> *   The penalty still influences how parameters are updated, but it is not part of the prediction metric.
> *   This separation is deliberate: during evaluation, we care about prediction performance, while during training we add regularization to influence how the model learns.

*   `optimizer.step()` applies parameter updates using the gradients from `backward()` and the optimizer's weight-decay rule..

## 5. Connection to ML systems

*   Most concise PyTorch implementations pass `weight_decay` to the optimizer.
*   Some advanced optimizers handle weight decay slightly differently, so optimizer documentation matters.

## 6. Common confusion points

- Built-in weight decay is an optimizer setting, not a separate visible loss term.
- Check whether the optimizer applies weight decay to all parameters or selected groups. For example:
    ```
    optimizer = torch.optim.AdamW([
        {"params": model.layer1.parameters(), "weight_decay": 0.01},
        {"params": model.layer2.parameters(), "weight_decay": 0} # decay skipped for 2nd layer
    ], lr=0.001)
    ```
> * Embedding parameters are sometimes excluded from weight decay because shrinking embedding vectors can interfere with the learned semantic representations.
> * `LayerNorm` (normalizes features within each individual example) and `BatchNorm` (normalizes using statistics across examples in a mini-batch) are also actively opt-out of weight decay because their parameters control the scaling and shifting of activations (values through network) rather than feature importance.
- Bias parameters are often excluded from weight decay since they shift predictions rather than control feature influence.
- Concise code still follows zero-grad, loss, backward, step.

# 3.7.5 Summary

## 1. Intuition

*   Weight decay is regularization that discourages large weights.
*   It can be written manually as an L2 penalty or requested through optimizer settings.



## 2. Why this exists

*   The goal is better generalization, especially when the model can overfit training noise.

## 3. Examples

*   Compare the two implementation styles.

In [ ]:
styles = {
    "scratch": "add wd * l2_penalty(w) to loss",
    "concise": "pass weight_decay to optimizer",
}

## 4. Step-by-step breakdown

*   The scratch version makes the penalty visible in the loss.

*   The concise version moves the penalty into the optimizer.

*   Both are attempts to control model complexity by discouraging large weights.



## 5. Connection to ML systems

*   Weight decay is one of the first regularization techniques used in deep learning systems.
*   Later chapters add more regularization tools.

## 6. Common confusion points

- Regularization improves expected behavior; it does not guarantee better validation loss every time.
- Too much weight decay can cause underfitting.
- Weight decay strength is a hyperparameter.
- Always compare training and validation performance.

# 3.7.6 Exercises

## 1. Intuition

*   These exercises practice adding and interpreting weight penalties.

## 2. Why this exists

*   The goal is to connect the regularization coefficient, weight size, and objective value.

## 3. Examples

*   Exercise 1: compute L2 penalty for a different weight vector.

In [11]:
def l2_penalty(w):
  return 0.5 * torch.sum(w ** 2)

w_test = torch.tensor([1.0, -2.0, 2.0])
l2_penalty(w_test) # (1^^2 + -2^^2 + 2^^2)*0.5 = (1+4+4)*0.5 = 4.5

tensor(4.5000)

*   Exercise 2: compare regularized losses with two penalty strengths.

In [12]:
data_loss = torch.tensor(1.5)
penalty = l2_penalty(w_test)
loss_small = data_loss + 0.01 * penalty
loss_large = data_loss + 1 * penalty
loss_small, loss_large # 1.5+0.01*4.5, 1.5+1*4.5, or 1.545, 6.0

(tensor(1.5450), tensor(6.))

## 4. Step-by-step breakdown

*   Exercise 1 checks the L2 penalty formula.
*   Exercise 2 checks that larger regularization strength increases the objective more for the same weights.
> *   The model would therefore feel more pressure to reduce weight size.



## 5. Connection to ML systems

*   This prepares you to tune weight decay using validation performance rather than guessing.

## 6. Common confusion points

- A larger penalty coefficient does not always mean better generalization (e.g. can underfit).
- The same coefficient can behave differently with different loss scales.
- Regularization changes training dynamics.
- Tune weight decay on validation data, not test data.